# 🪶 Automated Feather Measurement and Analysis Pipeline
This notebook runs a **minimal slice** of the full feather analysis workflow with **zero manual hand-offs** for the core extraction path. From one control cell, you dispatch distributed segmentation, pull outputs, and run downstream color-pattern analysis.

It keeps heavy model execution and dataset storage on the cluster while this notebook acts as the control plane. This preserves the lightweight interactive experience without mirroring the entire image corpus locally.


## Step 1: Laboratory Equipment Setup (Environment & Cluster Configuration)
The analysis environment is initialized by configuring cluster endpoints, Celery broker/backend connectivity, and path settings for remote input/output directories.

Use environment variables to override hosts, keys, and remote paths without editing code cells.


In [ ]:
import os
import shlex
import socket
import subprocess
import time
from pathlib import Path
from urllib.parse import urlparse

from celery import Celery
from IPython.display import display
from PIL import Image

# Cluster + path config (override with env vars as needed).
HEAD_HOST = os.getenv('FEATHER_HEAD_HOST', '10.0.0.148')
SSH_USER = os.getenv('FEATHER_SSH_USER', 'openteams')

# Key resolution: explicit env override first, then common local key names.
key_candidates = [
    os.path.expanduser(os.getenv('FEATHER_SSH_KEY', '')),
    os.path.expanduser('~/.ssh/ubuntu-mac-openteams-admin'),
    os.path.expanduser('~/.ssh/ubuntu-mac-cluster_user-admin'),
]
KEY_PATH = next((k for k in key_candidates if k and os.path.exists(k)), '')

REMOTE_REPO_DIR = os.getenv('FEATHER_REMOTE_REPO_DIR', '~/Feather_Molt_Project')
REMOTE_INPUT_DIR = os.getenv('FEATHER_REMOTE_INPUT_DIR', f'{REMOTE_REPO_DIR}/data/raw')
REMOTE_OUTPUT_DIR = os.getenv('FEATHER_REMOTE_OUTPUT_DIR', f'{REMOTE_REPO_DIR}/data/processed')
CLUSTER_HOSTS = [h.strip() for h in os.getenv('FEATHER_CLUSTER_HOSTS', '10.0.0.148,10.0.0.63,10.0.0.19,10.0.0.118').split(',') if h.strip()]

# Celery broker/backend reachable from this notebook host.
BROKER_URL = os.getenv('BROKER_URL', f'redis://{HEAD_HOST}:6379/0')
RESULT_BACKEND = os.getenv('RESULT_BACKEND', '')
if RESULT_BACKEND:
    os.environ['RESULT_BACKEND'] = RESULT_BACKEND
os.environ['BROKER_URL'] = BROKER_URL

def assert_tcp_reachable(url: str, name: str, timeout: float = 2.0):
    parsed = urlparse(url)
    host = parsed.hostname
    port = parsed.port or 6379
    if not host:
        raise RuntimeError(f'{name} URL is invalid: {url}')
    s = socket.socket()
    s.settimeout(timeout)
    try:
        s.connect((host, port))
    except Exception as exc:
        raise RuntimeError(
            f'{name} not reachable at {host}:{port} from this notebook host. '
            f'Original error: {type(exc).__name__}: {exc}'
        ) from exc
    finally:
        s.close()

assert_tcp_reachable(BROKER_URL, 'BROKER_URL')

# Use result backend only if explicitly set and reachable.
backend = None
if RESULT_BACKEND:
    assert_tcp_reachable(RESULT_BACKEND, 'RESULT_BACKEND')
    backend = RESULT_BACKEND

celery_app = Celery('feather_pipeline', broker=BROKER_URL, backend=backend)

LOCAL_PREVIEW_DIR = Path('./.minimal_slice_preview').resolve()
LOCAL_PREVIEW_DIR.mkdir(parents=True, exist_ok=True)

print('HEAD_HOST =', HEAD_HOST)
print('SSH_USER =', SSH_USER)
print('KEY_PATH =', KEY_PATH or '(none found; set FEATHER_SSH_KEY)')
print('CLUSTER_HOSTS =', CLUSTER_HOSTS)
print('BROKER_URL =', BROKER_URL)
print('RESULT_BACKEND =', RESULT_BACKEND or '(disabled)')
print('REMOTE_INPUT_DIR =', REMOTE_INPUT_DIR)
print('REMOTE_OUTPUT_DIR =', REMOTE_OUTPUT_DIR)
print('LOCAL_PREVIEW_DIR =', LOCAL_PREVIEW_DIR)


## Step 2: Remote Inventory and Transport Helpers
This step defines cluster-safe SSH/SCP helper functions for remote image discovery and artifact retrieval.

These helpers are the notebook-side I/O layer for the distributed pipeline: they let you browse source images and fetch processed outputs directly from remote storage.


In [ ]:
def _ssh_base_cmd(host: str):
    cmd = [
        'ssh',
        '-o', 'StrictHostKeyChecking=no',
        '-o', 'IdentitiesOnly=yes',
        '-o', 'IdentityAgent=none',
        '-o', 'BatchMode=yes',
        '-o', 'PreferredAuthentications=publickey',
    ]
    if not KEY_PATH:
        raise RuntimeError('No SSH key found. Set FEATHER_SSH_KEY to a readable private key path.')
    cmd += ['-i', KEY_PATH]
    cmd += [f'{SSH_USER}@{host}']
    return cmd


def _remote_shell_path(path: str) -> str:
    # Keep ~ expansion on the remote shell while still handling spaces safely.
    if path == '~':
        return '"$HOME"'
    if path.startswith('~/'):
        return '"$HOME/' + path[2:] + '"'
    return shlex.quote(path)


def ssh_lines(host: str, remote_cmd: str):
    out = subprocess.check_output([*_ssh_base_cmd(host), remote_cmd], text=True)
    return [line.strip() for line in out.splitlines() if line.strip()]


def list_remote_images(input_host: str = HEAD_HOST):
    d = _remote_shell_path(REMOTE_INPUT_DIR)
    cmd = f"find {d} -maxdepth 1 -type f \\( -iname '*.jpg' -o -iname '*.jpeg' \\) | sort"
    return ssh_lines(input_host, cmd)


def list_remote_outputs(host: str):
    d = _remote_shell_path(REMOTE_OUTPUT_DIR)
    cmd = f"find {d} -maxdepth 1 -type f -iname '*.jpg' | sort"
    return ssh_lines(host, cmd)


def fetch_remote_file(host: str, remote_path: str, local_dir: Path):
    local_dir.mkdir(parents=True, exist_ok=True)
    local_file = local_dir / Path(remote_path).name
    remote_cmd = f"cat {shlex.quote(remote_path)}"
    with open(local_file, 'wb') as f:
        subprocess.check_call([*_ssh_base_cmd(host), remote_cmd], stdout=f)


## Step 3: Minimal Slice Dispatch (Single-Image Distributed Processing)
A single source image is selected and submitted as a distributed task. Worker-side execution performs zero-shot feather extraction (Grounding DINO + SAM), metadata handling, and output materialization.

**Integration Note (Roboflow Training Handoff):**
This dispatch point is also where a custom-trained detector/segmenter can be swapped in. The generated overlays and crops from this workflow are suitable handoff artifacts for dataset curation and model fine-tuning pipelines (for example via Roboflow + YOLO).


In [ ]:
remote_images = list_remote_images(HEAD_HOST)
print(f'Found {len(remote_images)} remote images on {HEAD_HOST}.')
if not remote_images:
    raise RuntimeError('No remote images found. Check FEATHER_REMOTE_INPUT_DIR.')

SAMPLE_INDEX = 0  # Change this if you want a different source image.
image_path = remote_images[SAMPLE_INDEX]
source_stem = Path(image_path).stem
print('Selected remote source:', image_path)

# Fire-and-forget dispatch; completion is detected by output polling in Step 4.
async_result = celery_app.send_task(
    'src.celery_tasks.process_image',
    args=[image_path, REMOTE_OUTPUT_DIR],
    ignore_result=True,
)
print('Task dispatched. ID:', async_result.id)


## Step 4: Retrieval and Visual Review
After dispatch, the notebook polls cluster nodes for matching artifacts, fetches them to a local preview directory, and renders bounding-box overlays plus extracted feather crops inline.

This provides fast QA feedback before promoting outputs to larger runs or training datasets.


In [ ]:
prefix = f"{source_stem}_"
matching = []

WAIT_TIMEOUT_SECONDS = int(os.getenv('FEATHER_WAIT_TIMEOUT_SECONDS', '180'))
POLL_SECONDS = float(os.getenv('FEATHER_POLL_SECONDS', '3'))
deadline = time.time() + WAIT_TIMEOUT_SECONDS

while time.time() < deadline and not matching:
    for host in CLUSTER_HOSTS:
        try:
            out_files = list_remote_outputs(host)
        except Exception as exc:
            print(f'Skipping host {host}: {exc}')
            continue

        host_matches = []
        for p in out_files:
            name = Path(p).name
            if name.startswith(prefix) and ('_Feather_' in name or '_BoundingBoxes' in name):
                host_matches.append(p)

        if host_matches:
            print(f'Found {len(host_matches)} segment(s) on {host}.')
            matching = [(host, p) for p in host_matches]
            break

    if not matching:
        print('Waiting for output files...')
        time.sleep(POLL_SECONDS)

if not matching:
    raise RuntimeError(
        f'No matching segment outputs found within {WAIT_TIMEOUT_SECONDS}s. '
        'Check worker status and remote output path.'
    )

preview_dir = LOCAL_PREVIEW_DIR / source_stem
preview_dir.mkdir(parents=True, exist_ok=True)

for host, remote_file in matching:
    fetch_remote_file(host, remote_file, preview_dir)

local_files = sorted(preview_dir.glob('*.jpg'))
print(f'Fetched {len(local_files)} files into {preview_dir}')

bbox_files = [f for f in local_files if '_BoundingBoxes' in f.name]
feather_files = [f for f in local_files if '_Feather_' in f.name]

for f in bbox_files:
    print(f'\n=== DETECTION BOUNDING BOXES ===')
    print(f.name)
    display(Image.open(f))

for f in feather_files:
    print(f'\n=== EXTRACTED FEATHER ===')
    print(f.name)
    display(Image.open(f))


## Step 5: Color Pattern Analysis (Native R "pavo" Subprocess)
The extracted feather is routed into `pavo` to classify color patches and compute adjacency statistics for quantitative pattern analysis.

**Understanding the Output:**
- **Classified image:** reduces natural coloration into dominant pigment groups.
- **`adj_stats`:** measures neighborhood structure (how often class-to-class boundaries occur), which quantifies barring, spotting, and mottling.


In [ ]:
import subprocess
import os
import sys
import matplotlib.pyplot as plt
import cv2
import numpy as np

# Select the first feather from the fetched files to analyze
feather_files = [f for f in local_files if "_Feather_" in f.name]
if not feather_files:
    raise RuntimeError("No feather crops found to analyze.")
crop_path = str(feather_files[0])

r_script = f"""
suppressMessages(library(pavo))

cat('Analyzing segmented crop directly in R: {crop_path}\n')

# 1. LOAD IMAGE
feather_img <- getimg('{crop_path}')

# 2. CLASSIFY BIOLOGICAL PATCHES
classified_feather <- classify(feather_img, kcols = 3)

# 3. EXPORT RAW MATRIX DATA FOR PYTHON
# Bypassing R's graphical plotting to prevent dimension stretching!
write.table(attr(classified_feather, "classRGB"), "class_colors.csv", row.names=FALSE, col.names=FALSE, sep=",")
write.table(as.matrix(classified_feather), "class_matrix.csv", row.names=FALSE, col.names=FALSE, sep=",")

# 4. OUTPUT ADJACENCY STATISTICS
adj_stats <- adjacent(classified_feather, xscale = 1)
print(adj_stats)
"""

env = os.environ.copy()
if sys.platform == "darwin":
    env['R_HOME'] = "/Library/Frameworks/R.framework/Resources"
    env['PATH'] = env['R_HOME'] + "/bin:" + env.get('PATH', "")

print("Executing pavo R script...")
try:
    result = subprocess.run(['Rscript', '-e', r_script], capture_output=True, text=True, env=env)
    print(result.stdout)
    if result.stderr:
        print("R stderr:", result.stderr)
except FileNotFoundError:
    print("Rscript not found on this system. Skipping pavo analysis.")

# ---------------------------------------------------------
# RENDER WITH MATPLOTLIB FOR PERFECT DISPLAY
# ---------------------------------------------------------
if os.path.exists('class_matrix.csv') and os.path.exists('class_colors.csv'):
    # Load original un-stretched image directly from Python
    img1 = cv2.imread(crop_path)[:, :, ::-1] # BGR to RGB
    
    # Reconstruct the classified image manually
    matrix = np.loadtxt('class_matrix.csv', delimiter=',')
    colors = np.loadtxt('class_colors.csv', delimiter=',') # R, G, B floats
    
    # Create empty image
    img2 = np.zeros((matrix.shape[0], matrix.shape[1], 3))
    
    # Map classes to colors (Classes are 1, 2, 3 in R)
    for i in range(1, len(colors) + 1):
        img2[matrix == i] = colors[i-1]
        
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 10))
    
    ax1.imshow(img1)
    ax1.set_title("1. Segmented Crop", fontsize=14, pad=15)
    ax1.axis('off')
    
    ax2.imshow(img2)
    ax2.set_title("2. pavo k=3 Color Classification", fontsize=14, pad=15)
    ax2.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Clean up temp files
    os.remove('class_matrix.csv')
    os.remove('class_colors.csv')


## Step 6: Extracting Biological Traits and Visualizing Focus (BioCLIP 2.5 Embeddings & Saliency Heatmap)
This step converts a feather crop into a numeric feature embedding using BioCLIP and computes a lightweight saliency map for visual explainability.

Use this as an interpretability/feature-extraction checkpoint before any downstream modeling.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Reuse fetched feather crops from Step 4.
feather_files = [f for f in local_files if '_Feather_' in f.name]
if not feather_files:
    raise RuntimeError('No feather crops found. Run Step 4 first.')

crop_path = str(feather_files[0])
print('BioCLIP input:', crop_path)

try:
    import torch
    import open_clip
    from PIL import Image
except Exception as exc:  # noqa: BLE001
    raise RuntimeError(
        'BioCLIP dependencies missing. Install with: pip install open_clip_torch torch torchvision'
    ) from exc

device = 'mps' if getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available() else 'cpu'
print('device =', device)

model, _, preprocess = open_clip.create_model_and_transforms(
    'ViT-H-14', pretrained='hf-hub:imageomics/bioclip-2.5'
)
model = model.to(device).eval()
tokenizer = open_clip.get_tokenizer('ViT-H-14')

img_pil = Image.open(crop_path).convert('RGB')
img_tensor = preprocess(img_pil).unsqueeze(0).to(device)
img_tensor.requires_grad_(True)

text = tokenizer(['bird feather morphology']).to(device)

with torch.no_grad():
    text_features = model.encode_text(text)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

image_features = model.encode_image(img_tensor)
image_features = image_features / image_features.norm(dim=-1, keepdim=True)
score = (image_features @ text_features.T).squeeze()
score.backward()

embedding = image_features.detach().cpu().numpy().reshape(-1)
print('Embedding dimension:', embedding.shape[0])
print('Text-image similarity score:', float(score.detach().cpu().item()))

grad = img_tensor.grad.detach().abs().mean(dim=1).squeeze().cpu().numpy()
grad = (grad - grad.min()) / (grad.max() - grad.min() + 1e-8)

img_np = np.array(img_pil)
heat = np.uint8(255 * grad)
heat = np.array(Image.fromarray(heat).resize((img_np.shape[1], img_np.shape[0])))

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(img_np)
plt.title('Feather Crop')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(img_np)
plt.imshow(heat, alpha=0.45, cmap='inferno')
plt.title('BioCLIP Saliency Overlay')
plt.axis('off')
plt.tight_layout()
plt.show()


## Optional Step 7: Roboflow Training Workflow (Dataset Curation + YOLO Fine-Tuning)
Use this section to convert reviewed extraction outputs into a curated training dataset and fine-tune a custom segmentation model.

Suggested workflow:
1. Curate overlays/crops from high-quality runs.
2. Upload and correct labels in Roboflow.
3. Export YOLO-format segmentation data to `../data/autolabel/`.
4. Run the fine-tuning cell below.


In [ ]:
from pathlib import Path

DATASET_YAML = Path('../data/autolabel/dataset.yaml')

if not DATASET_YAML.exists():
    print('Roboflow export not found at ../data/autolabel/dataset.yaml')
    print('Export your curated dataset (YOLO format) to ../data/autolabel/ and re-run this cell.')
else:
    try:
        from ultralytics import YOLO
    except Exception as exc:  # noqa: BLE001
        print(f'Ultralytics import failed: {exc}')
        print('Install with: pip install ultralytics')
    else:
        model = YOLO('yolov12n-seg.pt')
        print('Starting YOLOv12 segmentation fine-tuning...')
        _ = model.train(
            data=str(DATASET_YAML),
            epochs=100,
            batch=16,
            device='mps',
            project='../models',
            name='yolov12_feather_seg',
            plots=True,
        )
        print('Training complete. Best weights at ../models/yolov12_feather_seg/weights/best.pt')
